# Step 11 · Resolve Wikipedia references

Resolve Wikipedia reference links for the pathogen records selected in the saved V4 run. The association table is keyed by `pathogen_id`, preserving stable V4 taxonomy identity. Lookups checkpoint to CSV after each API batch and resume completed records on rerun. Only exact Wikipedia article resolution or exact GBIF species matches receive links.

Run `00_configure_run` and `03_acquire_pair_pubmed_records` first. The selected run is read from `data/v4/runs/<run-id>/resolved_config.json`; when the run selected pathogen groups, the full resolved pathogen selection is included. Set `FORCE_REFRESH = True` to repeat lookups for records in scope.

## Step 1 · Load the resolved run and taxonomy


In [1]:
import json, sys
from pathlib import Path
import pandas as pd
from IPython.display import display, Markdown

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'pathogen_atlas_v4').is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'pathogen_atlas_v4').is_dir():
    raise RuntimeError('Open this notebook from the Pathogen Atlas V4 repository.')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT / 'notebooks'))
from _shared import configure_notebook, configured_run_id, staging_root_for_config
RUN_ID = configured_run_id(PROJECT_ROOT)
ROOT, LOG, LOG_PATH = configure_notebook('11_resolve_wikipedia_references', RUN_ID)
RUN_PATH = ROOT / 'data/v4/runs' / RUN_ID / 'resolved_config.json'
if not RUN_PATH.exists():
    raise FileNotFoundError(f'Run notebook 00 first; resolved selection is missing: {RUN_PATH}')
resolved_run = json.loads(RUN_PATH.read_text(encoding='utf-8'))
ACCEPTED_PATH = ROOT / 'data/v4/runs' / RUN_ID / 'release/pathogenic_interest_index.parquet'
if not ACCEPTED_PATH.exists():
    raise FileNotFoundError(f'Run notebook 05 first; accepted Explorer profiles are missing: {ACCEPTED_PATH}')
pathogen_ids = set(pd.read_parquet(ACCEPTED_PATH, columns=['pathogen_id']).pathogen_id.astype(str))
TAXONOMY_ROOT = staging_root_for_config(ROOT, resolved_run['source_config']) / 'taxonomy'
RUN_TAXONOMY_ROOT = ROOT / 'data/v4/runs' / RUN_ID / 'taxonomy'
PATHOGEN_TABLE = RUN_TAXONOMY_ROOT / 'pathogens.parquet'
if not PATHOGEN_TABLE.exists():
    raise FileNotFoundError('Run taxonomy was not resolved; run 00_configure_run.ipynb first.')
pathogens = pd.read_parquet(PATHOGEN_TABLE, columns=['pathogen_id','pathogen_group','accepted_name','rank'])
scope = pathogens.loc[pathogens.pathogen_id.astype(str).isin(pathogen_ids) & pathogens['rank'].astype(str).str.casefold().eq('species')]
PATHOGEN_GROUPS = sorted(scope.pathogen_group.astype(str).unique())
OUTPUT_PATH = ROOT / 'data/v4/references/release/11_resolve_wikipedia_references.csv'
FORCE_REFRESH = False
existing = pd.read_csv(OUTPUT_PATH, keep_default_na=False) if OUTPUT_PATH.exists() else pd.DataFrame()
display(Markdown(f"**Run:** `{RUN_ID}` · **Groups:** {', '.join(PATHOGEN_GROUPS)} · **Accepted species profiles:** {scope.pathogen_id.nunique():,} · **Existing rows:** {len(existing):,} · **Force refresh:** {FORCE_REFRESH}"))
LOG.info('Reference lookup prepared: run=%s groups=%s species=%d existing=%d', RUN_ID, PATHOGEN_GROUPS, scope.pathogen_id.nunique(), len(existing))


2026-11-03 16:45 INFO Notebook session started at 2026-11-03T16:45:59.705249+00:00
2026-11-03 16:45 INFO Project root: /Users/f.costa/Documents/Projects/Repositories/ACTIVE/Pathogen Atlas/Pathogen Atlas/Pathogen Atlas Data Acquisition/version4
2026-11-03 16:45 INFO Run log: /Users/f.costa/Documents/Projects/Repositories/ACTIVE/Pathogen Atlas/Pathogen Atlas/Pathogen Atlas Data Acquisition/version4/data/v4/logs/human-parasites-2026-11-03/11_resolve_wikipedia_references.log


**Run:** `human-parasites-2026-11-03` · **Groups:** parasites · **Accepted species profiles:** 348 · **Existing rows:** 3 · **Force refresh:** False

2026-11-03 16:45 INFO Reference lookup prepared: run=human-parasites-2026-11-03 groups=['parasites'] species=348 existing=3


## Step 2 · Resolve and save Wikipedia associations

The lookup writes one row per pathogen ID, saves progress after every request batch, and reuses completed rows on rerun.

In [2]:
from pathogen_atlas_v4.references.links import resolve_wikipedia_links
service = 'Wikipedia'

links = resolve_wikipedia_links(PATHOGEN_TABLE, OUTPUT_PATH, PATHOGEN_GROUPS, force_refresh=FORCE_REFRESH, pathogen_ids=pathogen_ids)
found = links.match_status.eq('found')
display(Markdown(f"**Saved:** `{OUTPUT_PATH.relative_to(ROOT)}` · **Records:** {len(links):,} · **{service} references found:** {int(found.sum()):,} · **No exact reference:** {int((~found).sum()):,}"))
display(links.head(12))
LOG.info('Reference lookup complete: service=Wikipedia records=%d found=%d output=%s', len(links), int(found.sum()), OUTPUT_PATH)


Resolving 348 pathogen records; reusing 3 completed rows.
Resolved 25/348 pending records.
Resolved 50/348 pending records.
Resolved 75/348 pending records.
Resolved 100/348 pending records.
Resolved 125/348 pending records.
Resolved 150/348 pending records.
Resolved 175/348 pending records.
Resolved 200/348 pending records.
Resolved 225/348 pending records.
Resolved 250/348 pending records.
Resolved 275/348 pending records.
Resolved 300/348 pending records.
Resolved 325/348 pending records.
Resolved 348/348 pending records.
Saved 351 associations to /Users/f.costa/Documents/Projects/Repositories/ACTIVE/Pathogen Atlas/Pathogen Atlas/Pathogen Atlas Data Acquisition/version4/data/v4/references/release/11_resolve_wikipedia_references.csv


**Saved:** `data/v4/references/release/11_resolve_wikipedia_references.csv` · **Records:** 351 · **Wikipedia references found:** 130 · **No exact reference:** 221

,pathogen_id,pathogen_group,accepted_name,wikipedia_title,wikipedia_url,match_status,match_type
0,bacteria:09e849d5c374f331c85d,parasites,Chimaeribacter arupi,,,not_found,exact_title_query
1,bacteria:2112dd7bba0bead55eb1,parasites,Peribacillus simplex,,,not_found,exact_title_query
2,bacteria:56faf32d2989cafe0289,parasites,Pseudocitrobacter vendiensis,,,not_found,exact_title_query
3,bacteria:799700ac070f5c419589,parasites,Campylobacter armoricus,,,not_found,exact_title_query
4,bacteria:94149d472b9198eef145,parasites,Yersinia canariae,Yersinia canariae,https://en.wikipedia.org/wiki/Yersinia_canariae,found,exact_or_redirect
5,bacteria:bf17dab67ef0f6ca6ddc,parasites,Enterocloster clostridioformis,Enterocloster clostridioformis,https://en.wikipedia.org/wiki/Enterocloster_cl...,found,exact_or_redirect
6,bacteria:ec760985ce2241f77f1f,parasites,Vogesella urethralis,,,not_found,exact_title_query
7,fungi:7718fd80a56ca847cbc0,parasites,Aspergillus lentulus,Aspergillus lentulus,https://en.wikipedia.org/wiki/Aspergillus_lent...,found,exact_or_redirect
8,fungi:c5ac93664da523db6159,parasites,Trichosporon faecale,,,not_found,exact_title_query
9,ncbi:1001829,parasites,Procerovum varium,,,not_found,exact_title_query


2026-11-03 16:47 INFO Reference lookup complete: service=Wikipedia records=351 found=130 output=/Users/f.costa/Documents/Projects/Repositories/ACTIVE/Pathogen Atlas/Pathogen Atlas/Pathogen Atlas Data Acquisition/version4/data/v4/references/release/11_resolve_wikipedia_references.csv
